# Assignment-1: POS tagger with CRF

> Submitted by: Thein Kyaw Lwin

## Prep Docker for Linux Env

In [ ]:
FROM ubuntu:22.04

LABEL maintainer="Thein Kyaw Lwin" description="Ubuntu environment prepared for CRF learning assignment and Jupyter"

# Prevent interactive prompts during apt installs
ENV DEBIAN_FRONTEND=noninteractive

# Update and install essentials + build tools required for liblbfgs & crfsuite compilation
RUN apt-get update && apt-get upgrade -y && apt-get install -y \
    apt-utils \
    curl \
    wget \
    git \
    vim \
    nano \
    iputils-ping \
    unzip \
    jq \
    tree \
    sudo \
    software-properties-common \
    # Compilers and Build Tools
    build-essential \
    cmake \
    # Required for running ./autogen.sh in liblbfgs and crfsuite
    autoconf \
    automake \
    libtool \
    pkg-config \
    # Python & Pip
    python3 \
    python3-pip \
    python3-venv \
    && rm -rf /var/lib/apt/lists/*

# Install Jupyter Notebook and helper Python libraries
RUN pip3 install --no-cache-dir \
    notebook \
    jupyterlab \
    matplotlib \
    pandas \
    datasets \
    pyarrow

# Set python alias
RUN ln -s /usr/bin/python3 /usr/bin/python

# Expose Jupyter port
EXPOSE 8888

# Set working directory to project workspace
WORKDIR /workspace

# Container bash prompt
RUN echo 'export PS1="\[\033[01;32m\]crf@container\[\033[00m\]:\[\033[01;34m\]\w\[\033[00m\]\$ "' >> /root/.bashrc

CMD ["/bin/bash"]

## Install `liblbfgs`
```bash
git clone https://github.com/chokkan/liblbfgs
cd liblbfgs
./autogen.sh
./configure
make
make install
ldconfig
cd ..
```

In [1]:
!mkdir tools

In [10]:
%cd tools

/workspace/tools


In [11]:
!pwd

/workspace/tools


In [12]:
!git clone https://github.com/chokkan/liblbfgs

Cloning into 'liblbfgs'...
remote: Enumerating objects: 528, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 528 (delta 23), reused 31 (delta 21), pack-reused 487 (from 1)
Receiving objects: 100% (528/528), 163.52 KiB | 602.00 KiB/s, done.
Resolving deltas: 100% (326/326), done.


In [13]:
%cd liblbfgs

/workspace/tools/liblbfgs


In [14]:
!./autogen.sh

libtoolize: putting auxiliary files in '.'.
libtoolize: copying file './ltmain.sh'
libtoolize: putting macros in AC_CONFIG_MACRO_DIRS, 'm4'.
libtoolize: copying file 'm4/libtool.m4'
libtoolize: copying file 'm4/ltoptions.m4'
libtoolize: copying file 'm4/ltsugar.m4'
libtoolize: copying file 'm4/ltversion.m4'
libtoolize: copying file 'm4/lt~obsolete.m4'
configure.ac:30: installing './compile'
configure.ac:30: installing './config.guess'
configure.ac:30: installing './config.sub'
configure.ac:22: installing './install-sh'
configure.ac:22: installing './missing'
configure.ac:102: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
Makefile.am: installing './INSTALL'
lib/Makefile.am:24: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
lib/Makefile.am: installing './depcomp'
sample/Makefile.am:15: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
configure.ac:48: warning: The macro `AC_HEADER_STDC' is obsolete.
configure.ac:48:

In [15]:
!./configure

checking for a BSD-compatible install... /usr/bin/install -c
checking whether build environment is sane... yes
checking for a race-free mkdir -p... /usr/bin/mkdir -p
checking for gawk... no
checking for mawk... mawk
checking whether make sets $(MAKE)... yes
checking whether make supports nested variables... yes
checking whether to enable maintainer-specific portions of Makefiles... no
checking build system type... aarch64-unknown-linux-gnu
checking host system type... aarch64-unknown-linux-gnu
checking how to print strings... printf
checking whether make supports the include directive... yes (GNU style)
checking for gcc... gcc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether the compiler supports GNU C... yes
checking whether gcc accepts -g... yes
checking for gcc option to enable C11 feature

In [16]:
!make

make  all-recursive
make[1]: Entering directory '/workspace/tools/liblbfgs'
Making all in lib
make[2]: Entering directory '/workspace/tools/liblbfgs/lib'
/bin/bash ../libtool  --tag=CC   --mode=compile gcc -DHAVE_CONFIG_H -I. -I.. -I.. -I../include   -O3 -ffast-math  -Wall -O3 -ffast-math  -Wall -MT lbfgs.lo -MD -MP -MF .deps/lbfgs.Tpo -c -o lbfgs.lo lbfgs.c
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I.. -I.. -I../include -O3 -ffast-math -Wall -O3 -ffast-math -Wall -MT lbfgs.lo -MD -MP -MF .deps/lbfgs.Tpo -c lbfgs.c  -fPIC -DPIC -o .libs/lbfgs.o
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I.. -I.. -I../include -O3 -ffast-math -Wall -O3 -ffast-math -Wall -MT lbfgs.lo -MD -MP -MF .deps/lbfgs.Tpo -c lbfgs.c -o lbfgs.o >/dev/null 2>&1
mv -f .deps/lbfgs.Tpo .deps/lbfgs.Plo
/bin/bash ../libtool  --tag=CC   --mode=link gcc -O3 -ffast-math  -Wall -O3 -ffast-math  -Wall -no-undefined -release 1.10  -o liblbfgs.la -rpath /usr/local/lib lbfgs.lo  -lm 
libtool: link: gcc -shared  -fPIC -DPIC  .li

In [17]:
!make install

Making install in lib
make[1]: Entering directory '/workspace/tools/liblbfgs/lib'
make[2]: Entering directory '/workspace/tools/liblbfgs/lib'
 /usr/bin/mkdir -p '/usr/local/lib'
 /bin/bash ../libtool   --mode=install /usr/bin/install -c   liblbfgs.la '/usr/local/lib'
libtool: install: /usr/bin/install -c .libs/liblbfgs-1.10.so /usr/local/lib/liblbfgs-1.10.so
libtool: install: (cd /usr/local/lib && { ln -s -f liblbfgs-1.10.so liblbfgs.so || { rm -f liblbfgs.so && ln -s liblbfgs-1.10.so liblbfgs.so; }; })
libtool: install: /usr/bin/install -c .libs/liblbfgs.lai /usr/local/lib/liblbfgs.la
libtool: install: /usr/bin/install -c .libs/liblbfgs.a /usr/local/lib/liblbfgs.a
libtool: install: chmod 644 /usr/local/lib/liblbfgs.a
libtool: install: ranlib /usr/local/lib/liblbfgs.a
libtool: finish: PATH="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/sbin" ldconfig -n /usr/local/lib
----------------------------------------------------------------------
Libraries have been installed in

In [18]:
!ldconfig

In [19]:
!pwd

/workspace/tools/liblbfgs


In [20]:
%cd ..

/workspace/tools


## Install `crfsuite` (Use `--disable-sse2` on Mac ARM64)
```bash
git clone https://github.com/chokkan/crfsuite
cd crfsuite
./autogen.sh
./configure --disable-sse2
make
make install
ldconfig
cd ..
```

In [21]:
!git clone https://github.com/chokkan/crfsuite

Cloning into 'crfsuite'...
remote: Enumerating objects: 3314, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 3314 (delta 0), reused 5 (delta 0), pack-reused 3307 (from 1)
Receiving objects: 100% (3314/3314), 1.04 MiB | 420.00 KiB/s, done.
Resolving deltas: 100% (2274/2274), done.


In [22]:
%cd crfsuite

/workspace/tools/crfsuite


In [23]:
!./autogen.sh

libtoolize: putting auxiliary files in '.'.
libtoolize: copying file './ltmain.sh'
libtoolize: putting macros in AC_CONFIG_MACRO_DIRS, 'm4'.
libtoolize: copying file 'm4/libtool.m4'
libtoolize: copying file 'm4/ltoptions.m4'
libtoolize: copying file 'm4/ltsugar.m4'
libtoolize: copying file 'm4/ltversion.m4'
libtoolize: copying file 'm4/lt~obsolete.m4'
aclocal: warning: autoconf input should be named 'configure.ac', not 'configure.in'
autoheader: warning: autoconf input should be named 'configure.ac', not 'configure.in'
automake: warning: autoconf input should be named 'configure.ac', not 'configure.in'
configure.in:22: installing './compile'
configure.in:21: installing './config.guess'
configure.in:21: installing './config.sub'
configure.in:30: installing './install-sh'
configure.in:30: installing './missing'
configure.in:140: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
configure.in:140: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')

In [24]:
!./configure --disable-sse2

checking build system type... aarch64-unknown-linux-gnu
checking host system type... aarch64-unknown-linux-gnu
checking for gcc... gcc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether the compiler supports GNU C... yes
checking whether gcc accepts -g... yes
checking for gcc option to enable C11 features... none needed
checking whether gcc understands -c and -o together... yes
checking for stdio.h... yes
checking for stdlib.h... yes
checking for string.h... yes
checking for inttypes.h... yes
checking for stdint.h... yes
checking for strings.h... yes
checking for sys/stat.h... yes
checking for sys/types.h... yes
checking for unistd.h... yes
checking for wchar.h... yes
checking for minix/config.h... no
checking whether it is safe to define __EXTENSIONS__... yes
checking whether _XOPEN_SOURCE sho

In [25]:
!make

make  all-recursive
make[1]: Entering directory '/workspace/tools/crfsuite'
Making all in include
make[2]: Entering directory '/workspace/tools/crfsuite/include'
make[2]: Nothing to be done for 'all'.
make[2]: Leaving directory '/workspace/tools/crfsuite/include'
Making all in lib/cqdb
make[2]: Entering directory '/workspace/tools/crfsuite/lib/cqdb'
/bin/bash ../../libtool  --tag=CC   --mode=compile gcc -DHAVE_CONFIG_H -I. -I../.. -I../.. -I../../include -I. -I../.. -I../../include -I.  -I./include -O3 -fomit-frame-pointer -ffast-math -Winline -std=c99  -MT libcqdb_la-lookup3.lo -MD -MP -MF .deps/libcqdb_la-lookup3.Tpo -c -o libcqdb_la-lookup3.lo `test -f 'src/lookup3.c' || echo './'`src/lookup3.c
libtool: compile:  gcc -DHAVE_CONFIG_H -I. -I../.. -I../.. -I../../include -I. -I../.. -I../../include -I. -I./include -O3 -fomit-frame-pointer -ffast-math -Winline -std=c99 -MT libcqdb_la-lookup3.lo -MD -MP -MF .deps/libcqdb_la-lookup3.Tpo -c src/lookup3.c  -fPIC -DPIC -o .libs/libcqdb_la-lo

In [26]:
!make install

Making install in include
make[1]: Entering directory '/workspace/tools/crfsuite/include'
make[2]: Entering directory '/workspace/tools/crfsuite/include'
make[2]: Nothing to be done for 'install-exec-am'.
 /usr/bin/mkdir -p '/usr/local/include'
 /usr/bin/install -c -m 644 crfsuite.h crfsuite_api.hpp crfsuite.hpp '/usr/local/include'
make[2]: Leaving directory '/workspace/tools/crfsuite/include'
make[1]: Leaving directory '/workspace/tools/crfsuite/include'
Making install in lib/cqdb
make[1]: Entering directory '/workspace/tools/crfsuite/lib/cqdb'
make[2]: Entering directory '/workspace/tools/crfsuite/lib/cqdb'
 /usr/bin/mkdir -p '/usr/local/lib'
 /bin/bash ../../libtool   --mode=install /usr/bin/install -c   libcqdb.la '/usr/local/lib'
libtool: install: /usr/bin/install -c .libs/libcqdb-0.12.so /usr/local/lib/libcqdb-0.12.so
libtool: install: (cd /usr/local/lib && { ln -s -f libcqdb-0.12.so libcqdb.so || { rm -f libcqdb.so && ln -s libcqdb-0.12.so libcqdb.so; }; })
libtool: install: /u

In [27]:
!ldconfig

In [28]:
%cd ..

/workspace/tools


## Verify Installation
```bash
crfsuite -h
# or
which crfsuite
```

In [29]:
!crfsuite -h

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

USAGE: crfsuite <COMMAND> [OPTIONS]
    COMMAND     Command name to specify the processing
    OPTIONS     Arguments for the command (optional; command-specific)

COMMAND:
    learn       Obtain a model from a training set of instances
    tag         Assign suitable labels to given instances by using a model
    dump        Output a model in a plain-text format

For the usage of each command, specify -h option in the command argument.


In [30]:
!crfsuite learn -h

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

USAGE: crfsuite learn [OPTIONS] [DATA1] [DATA2] ...
Trains a model using training data set(s).

  DATA    file(s) corresponding to data set(s) for training; if multiple N files
          are specified, this utility assigns a group number (1...N) to the
          instances in each file; if a file name is '-', the utility reads a
          data set from STDIN

OPTIONS:
  -t, --type=TYPE       specify a graphical model (DEFAULT='1d'):
                        (this option is reserved for the future use)
      1d                    1st-order Markov CRF with state and transition
                            features; transition features are not conditioned
                            on observations
  -a, --algorithm=NAME  specify a training algorithm (DEFAULT='lbfgs')
      lbfgs                 L-BFGS with L1/L2 regularization
      l2sgd                 SGD with L2-regularization
      ap                    Averaged Perceptron
      

In [31]:
!crfsuite tag -h

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

USAGE: crfsuite tag [OPTIONS] [DATA]
Assign suitable labels to the instances in the data set given by a file (DATA).
If the argument DATA is omitted or '-', this utility reads a data from STDIN.
Evaluate the performance of the model on labeled instances (with -t option).

OPTIONS:
    -m, --model=MODEL   Read a model from a file (MODEL)
    -t, --test          Report the performance of the model on the data
    -r, --reference     Output the reference labels in the input data
    -p, --probability   Output the probability of the label sequences
    -i, --marginal      Output the marginal probabilitiy of items for their predicted label
    -l, --marginal-all  Output the marginal probabilities of items for all labels
    -q, --quiet         Suppress tagging results (useful for test mode)
    -h, --help          Show the usage of this command and exit


In [32]:
!which crfsuite

/usr/local/bin/crfsuite


## Data Prep for CRFsuite

https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt

https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt

In [39]:
%pwd

'/workspace/data'

In [40]:
!wget https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt

--2026-07-30 08:09:15--  https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt [following]
--2026-07-30 08:09:16--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9581544 (9.1M) [application/octet-stream]
Saving to: ‘mypos-ver.3.0.shuf.nopipe.txt’

mypos-ver.3.0.shuf. 100%[========

In [41]:
!wget https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt

--2026-07-30 08:09:22--  https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt [following]
--2026-07-30 08:09:23--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229758 (224K) [text/plain]
Saving to: ‘otest.1k.nopipe.txt’

otest.1k.nopipe.txt 100%[===================>] 224.37K  --.-KB/s    in 0.05s   

2026-07

In [42]:
!ls

mypos-ver.3.0.shuf.nopipe.txt  otest.1k.nopipe.txt


## Python script (prepare_data.py) to parse myPOS data to CRFsuite compatible format (.crfsuite)

```python
#!/usr/bin/env python3
"""
Data Preparation Script for Myanmar POS Tagging with CRFSuite.
Converts raw 'word/tag' space-delimited text into tab-separated CRFSuite feature format.
"""

import sys
import os

# Input and Output paths
INPUT_FILE = "data/mypos-ver.3.0.shuf.nopipe.txt"
TRAIN_OUTPUT = "data/train.crfsuite"
TEST_OUTPUT = "data/test.crfsuite"
SPLIT_RATIO = 0.8  # 80% train, 20% test

def is_myanmar_or_arabic_digit(text):
    """Check if the text consists of Myanmar digits (၀-၉) or Arabic digits (0-9)."""
    myanmar_digits = set("၀၁၂၃၄၅၆၇၈၉0123456789")
    return len(text) > 0 and all(char in myanmar_digits for char in text)

def is_punctuation(text):
    """Check if the text is a punctuation mark."""
    punc_chars = set("။၊()\\_'\"")
    return text in punc_chars

def extract_word_features(sentence, i):
    """
    Extract features for word at position i in the given sentence list [(word, tag), ...].
    """
    word, tag = sentence[i]
    
    # Target label (POS Tag) MUST be the FIRST element for CRFSuite
    features = [
        tag,
        f"w[0]={word}",
        f"pref1={word[:1]}",
        f"suff1={word[-1:]}",
        f"suff2={word[-2:] if len(word) >= 2 else word}",
        f"is_digit={is_myanmar_or_arabic_digit(word)}",
        f"is_punc={is_punctuation(word)}"
    ]
    
    # Previous word feature (w[-1])
    if i > 0:
        prev_word, _ = sentence[i - 1]
        features.append(f"w[-1]={prev_word}")
    else:
        features.append("__BOS__")
        
    # Next word feature (w[1])
    if i < len(sentence) - 1:
        next_word, _ = sentence[i + 1]
        features.append(f"w[1]={next_word}")
    else:
        features.append("__EOS__")
        
    return features

def parse_line_to_sentence(line):
    """
    Parse a single raw line into a list of (word, tag) tuples.
    Example line: 'ဒီ/adj ဆေး/n က/ppm'
    Output: [('ဒီ', 'adj'), ('ဆေး', 'n'), ('က', 'ppm')]
    """
    line = line.strip()
    if not line:
        return []
    
    sentence = []
    tokens = line.split()
    for token in tokens:
        if '/' in token:
            # Split from the rightmost '/' to handle words that might contain '/'
            word, tag = token.rsplit('/', 1)
            sentence.append((word, tag))
    return sentence

def main():
    print(f"Loading raw dataset from {INPUT_FILE}...")
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found!")
        sys.exit(1)
        
    sentences = []
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            sent = parse_line_to_sentence(line)
            if sent:
                sentences.append(sent)
                
    print(f"Total sentences loaded: {len(sentences)}")
    
    # Train / Test split
    split_index = int(len(sentences) * SPLIT_RATIO)
    train_sentences = sentences[:split_index]
    test_sentences = sentences[split_index:]
    
    print(f"Training set: {len(train_sentences)} sentences")
    print(f"Testing set:  {len(test_sentences)} sentences")
    
    # Write Training Data
    print(f"Writing CRFSuite training features to {TRAIN_OUTPUT}...")
    with open(TRAIN_OUTPUT, "w", encoding="utf-8") as f:
        for sent in train_sentences:
            for i in range(len(sent)):
                features = extract_word_features(sent, i)
                f.write("\t".join(features) + "\n")
            f.write("\n")  # Blank line between sentences
            
    # Write Testing Data
    print(f"Writing CRFSuite testing features to {TEST_OUTPUT}...")
    with open(TEST_OUTPUT, "w", encoding="utf-8") as f:
        for sent in test_sentences:
            for i in range(len(sent)):
                features = extract_word_features(sent, i)
                f.write("\t".join(features) + "\n")
            f.write("\n")  # Blank line between sentences
            
    print("Data preparation complete!")

if __name__ == "__main__":
    main()

```

In [43]:
%cd ..

/workspace


In [44]:
!ls

CRF-tutorial.ipynb  data	     thein_kyaw_lwin.assignment-1.ipynb
crf-docker.md	    model	     tools
crf.Dockerfile	    prepare_data.py


In [45]:
!python prepare_data.py

Loading raw dataset from data/mypos-ver.3.0.shuf.nopipe.txt...
Total sentences loaded: 43196
Training set: 34556 sentences
Testing set:  8640 sentences
Writing CRFSuite training features to data/train.crfsuite...
Writing CRFSuite testing features to data/test.crfsuite...
Data preparation complete!


In [46]:
!ls data/

mypos-ver.3.0.shuf.nopipe.txt  test.crfsuite
otest.1k.nopipe.txt	       train.crfsuite


In [48]:
!head -n 10 data/train.crfsuite

num	w[0]=၁၉၆၂	pref1=၁	suff1=၂	suff2=၆၂	is_digit=True	is_punc=False	__BOS__	w[1]=ခုနှစ်
n	w[0]=ခုနှစ်	pref1=ခ	suff1=်	suff2=စ်	is_digit=False	is_punc=False	w[-1]=၁၉၆၂	w[1]=ခန့်မှန်း
v	w[0]=ခန့်မှန်း	pref1=ခ	suff1=း	suff2=်း	is_digit=False	is_punc=False	w[-1]=ခုနှစ်	w[1]=သန်းခေါင်စာရင်း
n	w[0]=သန်းခေါင်စာရင်း	pref1=သ	suff1=း	suff2=်း	is_digit=False	is_punc=False	w[-1]=ခန့်မှန်း	w[1]=အရ
ppm	w[0]=အရ	pref1=အ	suff1=ရ	suff2=အရ	is_digit=False	is_punc=False	w[-1]=သန်းခေါင်စာရင်း	w[1]=လူဦးရေ
n	w[0]=လူဦးရေ	pref1=လ	suff1=ေ	suff2=ရေ	is_digit=False	is_punc=False	w[-1]=အရ	w[1]=၁၁၅၉၃၁
num	w[0]=၁၁၅၉၃၁	pref1=၁	suff1=၁	suff2=၃၁	is_digit=True	is_punc=False	w[-1]=လူဦးရေ	w[1]=ယောက်
part	w[0]=ယောက်	pref1=ယ	suff1=်	suff2=က်	is_digit=False	is_punc=False	w[-1]=၁၁၅၉၃၁	w[1]=ရှိ
v	w[0]=ရှိ	pref1=ရ	suff1=ိ	suff2=ှိ	is_digit=False	is_punc=False	w[-1]=ယောက်	w[1]=သည်
ppm	w[0]=သည်	pref1=သ	suff1=်	suff2=ည်	is_digit=False	is_punc=False	w[-1]=ရှိ	w[1]=။


In [50]:
!head -n 10 data/test.crfsuite

n	w[0]=လေကြောင်း	pref1=လ	suff1=း	suff2=်း	is_digit=False	is_punc=False	__BOS__	w[1]=စာပို့
v	w[0]=စာပို့	pref1=စ	suff1=့	suff2=ု့	is_digit=False	is_punc=False	w[-1]=လေကြောင်း	w[1]=စနစ်
n	w[0]=စနစ်	pref1=စ	suff1=်	suff2=စ်	is_digit=False	is_punc=False	w[-1]=စာပို့	w[1]=ဖြင့်
ppm	w[0]=ဖြင့်	pref1=ဖ	suff1=့	suff2=့်	is_digit=False	is_punc=False	w[-1]=စနစ်	w[1]=ပို့
v	w[0]=ပို့	pref1=ပ	suff1=့	suff2=ု့	is_digit=False	is_punc=False	w[-1]=ဖြင့်	w[1]=ပါ
part	w[0]=ပါ	pref1=ပ	suff1=ါ	suff2=ပါ	is_digit=False	is_punc=False	w[-1]=ပို့	w[1]=။
punc	w[0]=။	pref1=။	suff1=။	suff2=။	is_digit=False	is_punc=True	w[-1]=ပါ	__EOS__

adv	w[0]=အခုတလော	pref1=အ	suff1=ာ	suff2=ော	is_digit=False	is_punc=False	__BOS__	w[1]=အပြင်ဘက်
n	w[0]=အပြင်ဘက်	pref1=အ	suff1=်	suff2=က်	is_digit=False	is_punc=False	w[-1]=အခုတလော	w[1]=ကို


## Train the CRF model using train.crfsuite

In [52]:
!crfsuite learn -m model/pos_model.crfsuite data/train.crfsuite

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-07-30T08:17:01Z

Reading the data set(s)
[1] data/train.crfsuite
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 34557
Seconds required: 1.474

Statistics the data set(s)
Number of data sets (groups): 1
Number of instances: 34556
Number of items: 451891
Number of attributes: 65618
Number of labels: 15

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 111980
Seconds required: 0.342

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 999058.913901
Feature norm: 1.000000
Error norm: 193741.349574
Active features: 111980
Line search trials: 1
Line search step: 0.000004
Secon

In [53]:
!crfsuite tag -m model/pos_model.crfsuite -qt data/test.crfsuite

Performance by label (#match, #model, #ref) (precision, recall, F1):
    num: (1206, 1209, 1214) (0.9975, 0.9934, 0.9955)
    n: (23653, 25142, 24400) (0.9408, 0.9694, 0.9549)
    v: (15648, 16607, 16809) (0.9423, 0.9309, 0.9366)
    ppm: (17050, 17408, 17333) (0.9794, 0.9837, 0.9815)
    part: (26013, 26960, 26940) (0.9649, 0.9656, 0.9652)
    punc: (10776, 10783, 10776) (0.9994, 1.0000, 0.9997)
    conj: (3220, 3625, 3485) (0.8883, 0.9240, 0.9058)
    adj: (2462, 2863, 3199) (0.8599, 0.7696, 0.8123)
    adv: (1706, 1876, 2192) (0.9094, 0.7783, 0.8387)
    pron: (3944, 4069, 4125) (0.9693, 0.9561, 0.9627)
    tn: (1134, 1165, 1191) (0.9734, 0.9521, 0.9626)
    fw: (681, 696, 693) (0.9784, 0.9827, 0.9806)
    int: (126, 129, 142) (0.9767, 0.8873, 0.9299)
    sb: (38, 38, 47) (1.0000, 0.8085, 0.8941)
    abb: (55, 56, 80) (0.9821, 0.6875, 0.8088)
Macro-average precision, recall, F1: (0.957454, 0.905942, 0.928588)
Item accuracy: 107712 / 112626 (0.9564)
Instance accuracy: 5454 / 8640 (0.

## Python script (prepare_otest_data.py) to parse otest myPOS data to CRFsuite compatible format (.crfsuite)

```python
#!/usr/bin/env python3
"""
Data Preparation Script for Myanmar POS Tagging Evaluation (otest set).
Converts raw 'word/tag' space-delimited text into tab-separated CRFSuite feature format
for 100% of the otest dataset (no train/test split).
"""

import sys
import os

# Input and Output paths
INPUT_FILE = "data/otest.1k.nopipe.txt"
OUTPUT_FILE = "data/otest.crfsuite"

def is_myanmar_or_arabic_digit(text):
    """Check if the text consists of Myanmar digits (၀-၉) or Arabic digits (0-9)."""
    myanmar_digits = set("၀၁၂၃၄၅၆၇၈၉0123456789")
    return len(text) > 0 and all(char in myanmar_digits for char in text)

def is_punctuation(text):
    """Check if the text is a punctuation mark."""
    punc_chars = set("။၊()\\_'\"")
    return text in punc_chars

def extract_word_features(sentence, i):
    """
    Extract features for word at position i in the given sentence list [(word, tag), ...].
    """
    word, tag = sentence[i]
    
    # Target label (POS Tag) MUST be the FIRST element for CRFSuite
    features = [
        tag,
        f"w[0]={word}",
        f"pref1={word[:1]}",
        f"suff1={word[-1:]}",
        f"suff2={word[-2:] if len(word) >= 2 else word}",
        f"is_digit={is_myanmar_or_arabic_digit(word)}",
        f"is_punc={is_punctuation(word)}"
    ]
    
    # Previous word feature (w[-1])
    if i > 0:
        prev_word, _ = sentence[i - 1]
        features.append(f"w[-1]={prev_word}")
    else:
        features.append("__BOS__")
        
    # Next word feature (w[1])
    if i < len(sentence) - 1:
        next_word, _ = sentence[i + 1]
        features.append(f"w[1]={next_word}")
    else:
        features.append("__EOS__")
        
    return features

def parse_line_to_sentence(line):
    """
    Parse a single raw line into a list of (word, tag) tuples.
    Example line: 'ဒီ/adj ဆေး/n က/ppm'
    Output: [('ဒီ', 'adj'), ('ဆေး', 'n'), ('က', 'ppm')]
    """
    line = line.strip()
    if not line:
        return []
    
    sentence = []
    tokens = line.split()
    for token in tokens:
        if '/' in token:
            # Split from the rightmost '/' to handle words that might contain '/'
            word, tag = token.rsplit('/', 1)
            sentence.append((word, tag))
    return sentence

def main():
    print(f"Loading raw test dataset from {INPUT_FILE}...")
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found!")
        sys.exit(1)
        
    sentences = []
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            sent = parse_line_to_sentence(line)
            if sent:
                sentences.append(sent)
                
    print(f"Total sentences loaded: {len(sentences)}")
    
    # Write 100% of sentences to output file
    print(f"Writing CRFSuite test features to {OUTPUT_FILE}...")
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for sent in sentences:
            for i in range(len(sent)):
                features = extract_word_features(sent, i)
                f.write("\t".join(features) + "\n")
            f.write("\n")  # Blank line between sentences
            
    print(f"Data preparation complete! Processed {len(sentences)} sentences into {OUTPUT_FILE}.")

if __name__ == "__main__":
    main()

```

In [54]:
%pwd

'/workspace'

In [55]:
!python prepare_otest_data.py

Loading raw test dataset from data/otest.1k.nopipe.txt...
Total sentences loaded: 1000
Writing CRFSuite test features to data/otest.crfsuite...
Data preparation complete! Processed 1000 sentences into data/otest.crfsuite.


In [56]:
!head -n 10 data/otest.crfsuite

tn	w[0]=တစ်	pref1=တ	suff1=်	suff2=စ်	is_digit=False	is_punc=False	__BOS__	w[1]=ကိုက်
n	w[0]=ကိုက်	pref1=က	suff1=်	suff2=က်	is_digit=False	is_punc=False	w[-1]=တစ်	w[1]=ကို
ppm	w[0]=ကို	pref1=က	suff1=ု	suff2=ို	is_digit=False	is_punc=False	w[-1]=ကိုက်	w[1]=ဝမ်
n	w[0]=ဝမ်	pref1=ဝ	suff1=်	suff2=မ်	is_digit=False	is_punc=False	w[-1]=ကို	w[1]=ခုနှစ်ထောင်
tn	w[0]=ခုနှစ်ထောင်	pref1=ခ	suff1=်	suff2=င်	is_digit=False	is_punc=False	w[-1]=ဝမ်	w[1]=ပါ
part	w[0]=ပါ	pref1=ပ	suff1=ါ	suff2=ပါ	is_digit=False	is_punc=False	w[-1]=ခုနှစ်ထောင်	w[1]=။
punc	w[0]=။	pref1=။	suff1=။	suff2=။	is_digit=False	is_punc=True	w[-1]=ပါ	__EOS__

n	w[0]=မနှစ်	pref1=မ	suff1=်	suff2=စ်	is_digit=False	is_punc=False	__BOS__	w[1]=က
ppm	w[0]=က	pref1=က	suff1=က	suff2=က	is_digit=False	is_punc=False	w[-1]=မနှစ်	w[1]=သူ


In [57]:
!crfsuite tag -m model/pos_model.crfsuite -qt data/otest.crfsuite

Performance by label (#match, #model, #ref) (precision, recall, F1):
    num: (153, 153, 155) (1.0000, 0.9871, 0.9935)
    n: (2927, 3086, 3000) (0.9485, 0.9757, 0.9619)
    v: (1882, 1999, 2010) (0.9415, 0.9363, 0.9389)
    ppm: (2015, 2052, 2060) (0.9820, 0.9782, 0.9801)
    part: (3082, 3184, 3189) (0.9680, 0.9664, 0.9672)
    punc: (1270, 1270, 1270) (1.0000, 1.0000, 1.0000)
    conj: (376, 437, 411) (0.8604, 0.9148, 0.8868)
    adj: (284, 330, 366) (0.8606, 0.7760, 0.8161)
    adv: (202, 224, 262) (0.9018, 0.7710, 0.8313)
    pron: (454, 469, 476) (0.9680, 0.9538, 0.9608)
    tn: (136, 140, 142) (0.9714, 0.9577, 0.9645)
    fw: (85, 88, 87) (0.9659, 0.9770, 0.9714)
    int: (22, 22, 25) (1.0000, 0.8800, 0.9362)
    sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
    abb: (11, 11, 12) (1.0000, 0.9167, 0.9565)
Macro-average precision, recall, F1: (0.957869, 0.932712, 0.944347)
Item accuracy: 12902 / 13468 (0.9580)
Instance accuracy: 639 / 1000 (0.6390)
Elapsed time: 0.036993 [sec] (27032.1 [

In [60]:
!crfsuite tag -m model/pos_model.crfsuite data/otest.crfsuite > data/otest_predictions.txt

In [61]:
!head -n 10 data/otest_predictions.txt

tn
n
ppm
n
v
part
punc

n
ppm
